# Parallel sorting (chapter 14)

Runs every program of `ec2/sorting` on a Colab GPU. First choose **Runtime → Change runtime type → GPU (T4 is enough)**, then **Runtime → Run all**.

The theory is in `ec2/sorting/README.md`. The same `.cu` files also run on a laptop without a GPU: `clang++ -std=c++17 -O2 -pthread -x c++ -I../common FILE.cu -o FILE && ./FILE`.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

The compatibility header: plain CUDA under `nvcc`, a CPU emulation under `g++`/`clang++`.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

inline const char* cudaGetErrorString(cudaError_t) {
  return "no error";
}

inline cudaError_t cudaGetLastError() {
  return cudaSuccess;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## `bikeshare_nearest_bikes.cu`

In [ ]:
%%writefile bikeshare_nearest_bikes.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <utility>
#include <vector>

constexpr int BLOCK_THREADS = 128;
constexpr int ITEMS_PER_THREAD = 8;
constexpr int TILE = BLOCK_THREADS * ITEMS_PER_THREAD;
constexpr int UNSIGNED_BITS = 32;
constexpr unsigned PADDING_KEY = 0xFFFFFFFFu;
constexpr int RADIX_BITS = 4;
constexpr int BUCKETS = 1 << RADIX_BITS;

__device__ inline int coalesced_position(int item) {
  return item * BLOCK_THREADS + threadIdx.x;
}

__device__ inline int thread_owned_position(int item) {
  return threadIdx.x * ITEMS_PER_THREAD + item;
}

__device__ inline unsigned block_inclusive_scan(unsigned value, unsigned* buf, unsigned* total) {
  const int t = threadIdx.x;
  buf[t] = value;
  __syncthreads();

  int readHalf = 0;
  for (int stride = 1; stride < BLOCK_THREADS; stride *= 2) {
    const int writeHalf = 1 - readHalf;
    unsigned sum = buf[readHalf * BLOCK_THREADS + t];
    if (t >= stride) {
      sum += buf[readHalf * BLOCK_THREADS + t - stride];
    }
    buf[writeHalf * BLOCK_THREADS + t] = sum;
    readHalf = writeHalf;
    __syncthreads();
  }

  *total = buf[readHalf * BLOCK_THREADS + BLOCK_THREADS - 1];
  const unsigned inclusivePrefix = buf[readHalf * BLOCK_THREADS + t];
  __syncthreads();
  return inclusivePrefix;
}

__global__ void scan_block_kernel(const unsigned* in, unsigned* out, int n, unsigned* blockSums) {
  __shared__ unsigned tile[TILE];
  __shared__ unsigned buf[2 * BLOCK_THREADS];
  const int base = blockIdx.x * TILE;

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    tile[pos] = base + pos < n ? in[base + pos] : 0u;
  }
  __syncthreads();

  unsigned threadSum = 0;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    threadSum += tile[thread_owned_position(item)];
  }

  unsigned blockTotal;
  unsigned running = block_inclusive_scan(threadSum, buf, &blockTotal) - threadSum;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = thread_owned_position(item);
    const unsigned value = tile[pos];
    tile[pos] = running;
    running += value;
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (base + pos < n) {
      out[base + pos] = tile[pos];
    }
  }
  if (threadIdx.x == 0) {
    blockSums[blockIdx.x] = blockTotal;
  }
}

__global__ void add_block_offsets_kernel(unsigned* out, int n, const unsigned* blockOffsets) {
  const unsigned offset = blockOffsets[blockIdx.x];
  const int base = blockIdx.x * TILE;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int index = base + coalesced_position(item);
    if (index < n) {
      out[index] += offset;
    }
  }
}

constexpr size_t SCAN_WORKSPACE_SLACK = 2;

static size_t scanWorkspace(int n) {
  size_t words = SCAN_WORKSPACE_SLACK;
  for (int blocks = cdiv(n, TILE); blocks > 1; blocks = cdiv(blocks, TILE)) {
    const size_t sumsAndScannedSums = 2 * blocks;
    words += sumsAndScannedSums;
  }
  return words + SCAN_WORKSPACE_SLACK;
}

static void exclusiveScan(const unsigned* in, unsigned* out, int n, unsigned* workspace) {
  if (n <= 0) {
    return;
  }

  const int blocks = cdiv(n, TILE);
  unsigned* sums = workspace;
  unsigned* sumsScanned = sums + blocks;
  unsigned* deeperWorkspace = sumsScanned + blocks;
  launch(scan_block_kernel, dim3(blocks), dim3(BLOCK_THREADS), in, out, n, sums);
  if (blocks > 1) {
    exclusiveScan(sums, sumsScanned, blocks, deeperWorkspace);
    launch(add_block_offsets_kernel, dim3(blocks), dim3(BLOCK_THREADS), out, n, sumsScanned);
  }
}

__host__ __device__ inline unsigned digit_of(unsigned key, int shift) {
  return shift < UNSIGNED_BITS ? (key >> shift) & (BUCKETS - 1u) : 0u;
}

__host__ __device__ inline unsigned bit_of(unsigned key, int bit) {
  return bit < UNSIGNED_BITS ? (key >> bit) & 1u : 0u;
}

__host__ __device__ inline int bucket_major_index(int bucket, int block, int numBlocks) {
  return bucket * numBlocks + block;
}

__host__ __device__ inline int block_major_index(int block, int bucket) {
  return block * BUCKETS + bucket;
}

__global__ void radix_local_sort_kernel(const unsigned* keysIn,
                                        const unsigned* valsIn,
                                        int n,
                                        int shift,
                                        unsigned* keysTmp,
                                        unsigned* valsTmp,
                                        unsigned* counts,
                                        unsigned* localStart) {
  __shared__ unsigned keys_S[2][TILE];
  __shared__ unsigned vals_S[2][TILE];
  __shared__ unsigned buf[2 * BLOCK_THREADS];
  __shared__ unsigned bucketStart[BUCKETS];
  __shared__ unsigned bucketEnd[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    const bool isRealElement = pos < tileCount;
    keys_S[0][pos] = isRealElement ? keysIn[tileBase + pos] : PADDING_KEY;
    vals_S[0][pos] = isRealElement ? valsIn[tileBase + pos] : 0u;
  }
  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    bucketStart[d] = 0;
    bucketEnd[d] = 0;
  }
  __syncthreads();

  int current = 0;
  for (int bitInDigit = 0; bitInDigit < RADIX_BITS; ++bitInDigit) {
    const int bit = shift + bitInDigit;
    const int next = 1 - current;
    unsigned myBits[ITEMS_PER_THREAD];
    unsigned myOnes = 0;
    for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
      myBits[item] = bit_of(keys_S[current][thread_owned_position(item)], bit);
      myOnes += myBits[item];
    }

    unsigned totalOnes;
    unsigned onesBefore = block_inclusive_scan(myOnes, buf, &totalOnes) - myOnes;
    const unsigned zeros = TILE - totalOnes;
    for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
      const unsigned pos = thread_owned_position(item);
      const unsigned dst = myBits[item] ? zeros + onesBefore : pos - onesBefore;
      onesBefore += myBits[item];
      keys_S[next][dst] = keys_S[current][pos];
      vals_S[next][dst] = vals_S[current][pos];
    }
    __syncthreads();
    current = next;
  }

  const unsigned* sortedKeys = keys_S[current];
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = thread_owned_position(item);
    if (pos < tileCount) {
      const unsigned d = digit_of(sortedKeys[pos], shift);
      const bool startsBucket = pos == 0 || digit_of(sortedKeys[pos - 1], shift) != d;
      const bool endsBucket = pos == tileCount - 1 || digit_of(sortedKeys[pos + 1], shift) != d;
      if (startsBucket) {
        bucketStart[d] = pos;
      }
      if (endsBucket) {
        bucketEnd[d] = pos + 1;
      }
    }
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (pos < tileCount) {
      keysTmp[tileBase + pos] = keys_S[current][pos];
      valsTmp[tileBase + pos] = vals_S[current][pos];
    }
  }
  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    counts[bucket_major_index(d, blockIdx.x, gridDim.x)] = bucketEnd[d] - bucketStart[d];
    localStart[block_major_index(blockIdx.x, d)] = bucketStart[d];
  }
}

__global__ void radix_scatter_coalesced_kernel(const unsigned* keysTmp,
                                               const unsigned* valsTmp,
                                               int n,
                                               int shift,
                                               const unsigned* offsets,
                                               const unsigned* localStart,
                                               unsigned* keysOut,
                                               unsigned* valsOut) {
  __shared__ int localToGlobal[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    const int globalStart = offsets[bucket_major_index(d, blockIdx.x, gridDim.x)];
    const int bucketLocalStart = localStart[block_major_index(blockIdx.x, d)];
    localToGlobal[d] = globalStart - bucketLocalStart;
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (pos < tileCount) {
      const unsigned key = keysTmp[tileBase + pos];
      const int dst = localToGlobal[digit_of(key, shift)] + pos;
      keysOut[dst] = key;
      valsOut[dst] = valsTmp[tileBase + pos];
    }
  }
}

struct RadixSortScratch {
  unsigned* otherKeys;
  unsigned* otherVals;
  unsigned* tileSortedKeys;
  unsigned* tileSortedVals;
  unsigned* counts;
  unsigned* offsets;
  unsigned* localStart;
  unsigned* workspace;
};

static unsigned* allocateDeviceWords(size_t count) {
  unsigned* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(unsigned)));
  return pointer;
}

static RadixSortScratch allocateScratch(int n) {
  const int tableEntries = BUCKETS * cdiv(n, TILE);
  RadixSortScratch scratch;
  scratch.otherKeys = allocateDeviceWords(n);
  scratch.otherVals = allocateDeviceWords(n);
  scratch.tileSortedKeys = allocateDeviceWords(n);
  scratch.tileSortedVals = allocateDeviceWords(n);
  scratch.counts = allocateDeviceWords(tableEntries);
  scratch.offsets = allocateDeviceWords(tableEntries);
  scratch.localStart = allocateDeviceWords(tableEntries);
  scratch.workspace = allocateDeviceWords(scanWorkspace(tableEntries));
  return scratch;
}

static void freeScratch(RadixSortScratch& scratch) {
  CHECK(cudaFree(scratch.otherKeys));
  CHECK(cudaFree(scratch.otherVals));
  CHECK(cudaFree(scratch.tileSortedKeys));
  CHECK(cudaFree(scratch.tileSortedVals));
  CHECK(cudaFree(scratch.counts));
  CHECK(cudaFree(scratch.offsets));
  CHECK(cudaFree(scratch.localStart));
  CHECK(cudaFree(scratch.workspace));
}

static void radixSortPass(const unsigned* keysIn,
                          const unsigned* valsIn,
                          unsigned* keysOut,
                          unsigned* valsOut,
                          int n,
                          int shift,
                          RadixSortScratch& scratch) {
  const unsigned blocks = cdiv(n, TILE);
  launch(radix_local_sort_kernel,
         dim3(blocks),
         dim3(BLOCK_THREADS),
         keysIn,
         valsIn,
         n,
         shift,
         scratch.tileSortedKeys,
         scratch.tileSortedVals,
         scratch.counts,
         scratch.localStart);
  exclusiveScan(scratch.counts, scratch.offsets, BUCKETS * blocks, scratch.workspace);
  launch(radix_scatter_coalesced_kernel,
         dim3(blocks),
         dim3(BLOCK_THREADS),
         scratch.tileSortedKeys,
         scratch.tileSortedVals,
         n,
         shift,
         scratch.offsets,
         scratch.localStart,
         keysOut,
         valsOut);
}

static int radixPasses(int keyBits) {
  return (keyBits + RADIX_BITS - 1) / RADIX_BITS;
}

static void radixSort(unsigned* keys, unsigned* vals, int n, int keyBits) {
  RadixSortScratch scratch = allocateScratch(n);
  unsigned* keysIn = keys;
  unsigned* valsIn = vals;
  unsigned* keysOut = scratch.otherKeys;
  unsigned* valsOut = scratch.otherVals;
  for (int shift = 0; shift < keyBits; shift += RADIX_BITS) {
    radixSortPass(keysIn, valsIn, keysOut, valsOut, n, shift, scratch);
    std::swap(keysIn, keysOut);
    std::swap(valsIn, valsOut);
  }

  const bool resultIsInScratch = keysIn != keys;
  if (resultIsInScratch) {
    CHECK(cudaMemcpy(keys, keysIn, n * sizeof(unsigned), cudaMemcpyDeviceToDevice));
    CHECK(cudaMemcpy(vals, valsIn, n * sizeof(unsigned), cudaMemcpyDeviceToDevice));
  }
  CHECK(cudaGetLastError());
  CHECK(cudaDeviceSynchronize());
  freeScratch(scratch);
}

constexpr int DISTANCE_BITS = 14;
constexpr unsigned DISTANCE_MASK = (1u << DISTANCE_BITS) - 1;
constexpr unsigned DISTANCE_UNAVAILABLE = DISTANCE_MASK;
constexpr int LONGEST_OFFERED_DISTANCE = DISTANCE_UNAVAILABLE - 1;
constexpr int MIN_BATTERY_PERCENT = 30;
constexpr int NO_STATION = -1;

enum class BikeState { Docked, Riding, InRepair };

struct Position {
  int x;
  int y;
};

struct Bike {
  int id;
  BikeState state;
  Position position;
  int batteryPercent;
  int station;
};

struct RideRequest {
  int id;
  Position rider;
};

__host__ __device__ inline int walking_distance_metres(Position from, Position to) {
  const int dx = from.x - to.x;
  const int dy = from.y - to.y;
  return (dx < 0 ? -dx : dx) + (dy < 0 ? -dy : dy);
}

__host__ __device__ inline bool can_be_offered(const Bike& bike) {
  return bike.state == BikeState::Docked && bike.batteryPercent >= MIN_BATTERY_PERCENT;
}

__host__ __device__ inline unsigned offered_distance(const Bike& bike, const RideRequest& request) {
  if (!can_be_offered(bike)) {
    return DISTANCE_UNAVAILABLE;
  }
  const int metres = walking_distance_metres(request.rider, bike.position);
  return imin(metres, LONGEST_OFFERED_DISTANCE);
}

__host__ __device__ inline unsigned request_distance_key(int request, unsigned distance) {
  return (static_cast<unsigned>(request) << DISTANCE_BITS) | distance;
}

__global__ void distance_keys_kernel(const Bike* bikes,
                                     int numBikes,
                                     const RideRequest* requests,
                                     int numRequests,
                                     unsigned* keys,
                                     unsigned* vals) {
  const int pair = blockIdx.x * blockDim.x + threadIdx.x;
  if (pair < numRequests * numBikes) {
    const int request = pair / numBikes;
    const int bike = pair % numBikes;
    keys[pair] = request_distance_key(request, offered_distance(bikes[bike], requests[request]));
    vals[pair] = bikes[bike].id;
  }
}

#ifndef CPU_REQUESTS
#define CPU_REQUESTS 64
#endif

#ifndef CPU_BIKES
#define CPU_BIKES 256
#endif

constexpr int RUSH_REQUESTS = BY_BACKEND(1024, CPU_REQUESTS);
constexpr int FLEET_BIKES = BY_BACKEND(4096, CPU_BIKES);
constexpr int CANDIDATE_PAIRS = RUSH_REQUESTS * FLEET_BIKES;
constexpr int BIKES_OFFERED = 3;
constexpr int REQUESTS_SHOWN = 4;
constexpr int DISTANCE_THREADS_PER_BLOCK = 256;

constexpr int MAP_SIDE_METRES = 6000;
constexpr int STATION_COUNT = 48;
constexpr uint64_t STATION_SEED = 48;
constexpr int STATION_MARGIN_METRES = 300;
constexpr uint64_t SCENARIO_SEED = 7;
constexpr int HUNDRED_PERCENT = 100;
constexpr int DOCKED_PERCENT = 70;
constexpr int RIDING_PERCENT = 22;
constexpr int FULL_BATTERY_PERCENT = 100;

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int below(int n) {
    return static_cast<int>(next() % static_cast<uint64_t>(n));
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

static int bitsNeeded(unsigned maxValue) {
  int bits = 0;
  while (bits < UNSIGNED_BITS && (maxValue >> bits) != 0) {
    ++bits;
  }
  return bits;
}

static Position randomPosition(SplitMix64& rng, int marginMetres) {
  const int span = MAP_SIDE_METRES - 2 * marginMetres;
  Position position;
  position.x = marginMetres + rng.below(span);
  position.y = marginMetres + rng.below(span);
  return position;
}

static std::vector<Position> makeStationPositions() {
  SplitMix64 rng(STATION_SEED);
  std::vector<Position> stations(STATION_COUNT);
  for (Position& station : stations) {
    station = randomPosition(rng, STATION_MARGIN_METRES);
  }
  return stations;
}

static BikeState pickState(SplitMix64& rng) {
  const int draw = rng.below(HUNDRED_PERCENT);
  if (draw < DOCKED_PERCENT) {
    return BikeState::Docked;
  }
  if (draw < DOCKED_PERCENT + RIDING_PERCENT) {
    return BikeState::Riding;
  }
  return BikeState::InRepair;
}

static Bike makeBike(SplitMix64& rng, int id, const std::vector<Position>& stations) {
  Bike bike;
  bike.id = id;
  bike.state = pickState(rng);
  bike.batteryPercent = rng.below(FULL_BATTERY_PERCENT + 1);
  if (bike.state == BikeState::Riding) {
    bike.station = NO_STATION;
    bike.position = randomPosition(rng, 0);
  } else {
    bike.station = rng.below(STATION_COUNT);
    bike.position = stations[bike.station];
  }
  return bike;
}

struct Scenario {
  std::vector<Bike> fleet;
  std::vector<RideRequest> rush;
};

static Scenario makeScenario() {
  const std::vector<Position> stations = makeStationPositions();
  SplitMix64 rng(SCENARIO_SEED);

  Scenario scenario;
  scenario.fleet.resize(FLEET_BIKES);
  for (int id = 0; id < FLEET_BIKES; ++id) {
    scenario.fleet[id] = makeBike(rng, id, stations);
  }
  scenario.rush.resize(RUSH_REQUESTS);
  for (int id = 0; id < RUSH_REQUESTS; ++id) {
    scenario.rush[id] = {id, randomPosition(rng, 0)};
  }
  return scenario;
}

struct RankedCandidates {
  std::vector<unsigned> keys;
  std::vector<unsigned> bikes;
  double ms;
};

static RankedCandidates rankOnDevice(const Scenario& scenario, int keyBits) {
  Bike* deviceFleet;
  RideRequest* deviceRush;
  CHECK(cudaMalloc(&deviceFleet, FLEET_BIKES * sizeof(Bike)));
  CHECK(cudaMalloc(&deviceRush, RUSH_REQUESTS * sizeof(RideRequest)));
  unsigned* deviceKeys = allocateDeviceWords(CANDIDATE_PAIRS);
  unsigned* deviceVals = allocateDeviceWords(CANDIDATE_PAIRS);
  CHECK(cudaMemcpy(
      deviceFleet, scenario.fleet.data(), FLEET_BIKES * sizeof(Bike), cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(deviceRush,
                   scenario.rush.data(),
                   RUSH_REQUESTS * sizeof(RideRequest),
                   cudaMemcpyHostToDevice));

  Timer timer;
  launch(distance_keys_kernel,
         dim3(cdiv(CANDIDATE_PAIRS, DISTANCE_THREADS_PER_BLOCK)),
         dim3(DISTANCE_THREADS_PER_BLOCK),
         deviceFleet,
         FLEET_BIKES,
         deviceRush,
         RUSH_REQUESTS,
         deviceKeys,
         deviceVals);
  radixSort(deviceKeys, deviceVals, CANDIDATE_PAIRS, keyBits);
  const double ms = timer.ms();

  RankedCandidates ranked{
      std::vector<unsigned>(CANDIDATE_PAIRS), std::vector<unsigned>(CANDIDATE_PAIRS), ms};
  const size_t pairBytes = CANDIDATE_PAIRS * sizeof(unsigned);
  CHECK(cudaMemcpy(ranked.keys.data(), deviceKeys, pairBytes, cudaMemcpyDeviceToHost));
  CHECK(cudaMemcpy(ranked.bikes.data(), deviceVals, pairBytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(deviceFleet));
  CHECK(cudaFree(deviceRush));
  CHECK(cudaFree(deviceKeys));
  CHECK(cudaFree(deviceVals));
  return ranked;
}

struct DistanceBike {
  unsigned distance;
  unsigned bike;
};

static std::vector<DistanceBike> rankOnHost(const Scenario& scenario) {
  std::vector<DistanceBike> ranking(CANDIDATE_PAIRS);
  for (int request = 0; request < RUSH_REQUESTS; ++request) {
    const auto first = ranking.begin() + static_cast<size_t>(request) * FLEET_BIKES;
    for (int bike = 0; bike < FLEET_BIKES; ++bike) {
      first[bike] = {offered_distance(scenario.fleet[bike], scenario.rush[request]),
                     static_cast<unsigned>(scenario.fleet[bike].id)};
    }
    std::stable_sort(first, first + FLEET_BIKES, [](const DistanceBike& a, const DistanceBike& b) {
      return a.distance < b.distance;
    });
  }
  return ranking;
}

static int requestOfKey(unsigned key) {
  return key >> DISTANCE_BITS;
}

static unsigned distanceOfKey(unsigned key) {
  return key & DISTANCE_MASK;
}

struct RankingChecks {
  bool offeredBikesOk;
  bool wholeRankingOk;
};

static RankingChecks compareRankings(const RankedCandidates& device,
                                     const std::vector<DistanceBike>& host) {
  RankingChecks checks{true, true};
  for (int pair = 0; pair < CANDIDATE_PAIRS; ++pair) {
    const int request = pair / FLEET_BIKES;
    const int rank = pair % FLEET_BIKES;
    const unsigned key = device.keys[pair];
    const bool same = requestOfKey(key) == request && distanceOfKey(key) == host[pair].distance &&
                      device.bikes[pair] == host[pair].bike;
    checks.wholeRankingOk = checks.wholeRankingOk && same;
    if (rank < BIKES_OFFERED) {
      checks.offeredBikesOk = checks.offeredBikesOk && same;
    }
  }
  return checks;
}

static void printOffer(const RankedCandidates& ranked, const Scenario& scenario, int request) {
  const Position rider = scenario.rush[request].rider;
  std::printf("  request %d at (%d m, %d m):", request, rider.x, rider.y);
  for (int rank = 0; rank < BIKES_OFFERED; ++rank) {
    const int pair = request * FLEET_BIKES + rank;
    const unsigned distance = distanceOfKey(ranked.keys[pair]);
    if (distance == DISTANCE_UNAVAILABLE) {
      break;
    }
    const Bike& bike = scenario.fleet[ranked.bikes[pair]];
    const char* separator = rank > 0 ? "," : "";
    std::printf("%s b%d at s%d %u m", separator, bike.id, bike.station, distance);
  }
  std::printf("\n");
}

static void printOffers(const RankedCandidates& ranked, const Scenario& scenario) {
  std::printf("nearest charged bikes offered to the first riders:\n");
  for (int request = 0; request < REQUESTS_SHOWN; ++request) {
    printOffer(ranked, scenario, request);
  }
}

static int countBikes(const std::vector<Bike>& fleet, BikeState state) {
  return std::count_if(fleet.begin(), fleet.end(), [state](const Bike& bike) {
    return bike.state == state;
  });
}

static int countDockedButLowBattery(const std::vector<Bike>& fleet) {
  return std::count_if(fleet.begin(), fleet.end(), [](const Bike& bike) {
    return bike.state == BikeState::Docked && !can_be_offered(bike);
  });
}

static void printUnavailableBikes(const std::vector<Bike>& fleet) {
  const int riding = countBikes(fleet, BikeState::Riding);
  const int inRepair = countBikes(fleet, BikeState::InRepair);
  const int lowBattery = countDockedButLowBattery(fleet);
  std::printf(
      "bikes that cannot be offered: %d of %d (riding %d, in repair %d, docked below %d %% "
      "battery %d)\n\n",
      riding + inRepair + lowBattery,
      FLEET_BIKES,
      riding,
      inRepair,
      MIN_BATTERY_PERCENT,
      lowBattery);
}

static void printTimings(double cpuMs, double deviceMs) {
  std::printf("%-44s %10s\n", "step", "time (ms)");
  std::printf("%-44s %10.2f\n", "distance + stable_sort per request (CPU)", cpuMs);
  std::printf("%-44s %10.2f\n", "distance map + one radix sort (device)", deviceMs);
}

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

int main() {
  const int keyBits = DISTANCE_BITS + bitsNeeded(RUSH_REQUESTS - 1);
  std::printf("Nearest bikes: k charged available bikes per rider request on %s\n", CUDA_BACKEND);
  std::printf("%d requests x %d bikes = %d candidate pairs, key = %d bits -> %d radix passes\n\n",
              RUSH_REQUESTS,
              FLEET_BIKES,
              CANDIDATE_PAIRS,
              keyBits,
              radixPasses(keyBits));

  const Scenario scenario = makeScenario();
  const RankedCandidates device = rankOnDevice(scenario, keyBits);

  Timer cpuTimer;
  const std::vector<DistanceBike> host = rankOnHost(scenario);
  const double cpuMs = cpuTimer.ms();

  const RankingChecks checks = compareRankings(device, host);

  printOffers(device, scenario);
  printUnavailableBikes(scenario.fleet);
  printTimings(cpuMs, device.ms);
  std::printf("\nk = %d nearest bikes equal reference:  %s\n",
              BIKES_OFFERED,
              passOrFail(checks.offeredBikesOk));
  std::printf("whole ranking equal reference:        %s\n", passOrFail(checks.wholeRankingOk));
  const bool ok = checks.offeredBikesOk && checks.wholeRankingOk;
  std::printf("%s\n", ok ? "ALL PASS" : "SOME TESTS FAILED");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_nearest_bikes.cu -o bikeshare_nearest_bikes && ./bikeshare_nearest_bikes

## `bikeshare_trip_percentiles.cu`

In [ ]:
%%writefile bikeshare_trip_percentiles.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <utility>
#include <vector>

constexpr int BLOCK_THREADS = 128;
constexpr int ITEMS_PER_THREAD = 8;
constexpr int TILE = BLOCK_THREADS * ITEMS_PER_THREAD;
constexpr int UNSIGNED_BITS = 32;
constexpr unsigned PADDING_KEY = 0xFFFFFFFFu;
constexpr int RADIX_BITS = 4;
constexpr int BUCKETS = 1 << RADIX_BITS;

__device__ inline int coalesced_position(int item) {
  return item * BLOCK_THREADS + threadIdx.x;
}

__device__ inline int thread_owned_position(int item) {
  return threadIdx.x * ITEMS_PER_THREAD + item;
}

__device__ inline unsigned block_inclusive_scan(unsigned value, unsigned* buf, unsigned* total) {
  const int t = threadIdx.x;
  buf[t] = value;
  __syncthreads();

  int readHalf = 0;
  for (int stride = 1; stride < BLOCK_THREADS; stride *= 2) {
    const int writeHalf = 1 - readHalf;
    unsigned sum = buf[readHalf * BLOCK_THREADS + t];
    if (t >= stride) {
      sum += buf[readHalf * BLOCK_THREADS + t - stride];
    }
    buf[writeHalf * BLOCK_THREADS + t] = sum;
    readHalf = writeHalf;
    __syncthreads();
  }

  *total = buf[readHalf * BLOCK_THREADS + BLOCK_THREADS - 1];
  const unsigned inclusivePrefix = buf[readHalf * BLOCK_THREADS + t];
  __syncthreads();
  return inclusivePrefix;
}

__global__ void scan_block_kernel(const unsigned* in, unsigned* out, int n, unsigned* blockSums) {
  __shared__ unsigned tile[TILE];
  __shared__ unsigned buf[2 * BLOCK_THREADS];
  const int base = blockIdx.x * TILE;

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    tile[pos] = base + pos < n ? in[base + pos] : 0u;
  }
  __syncthreads();

  unsigned threadSum = 0;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    threadSum += tile[thread_owned_position(item)];
  }

  unsigned blockTotal;
  unsigned running = block_inclusive_scan(threadSum, buf, &blockTotal) - threadSum;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = thread_owned_position(item);
    const unsigned value = tile[pos];
    tile[pos] = running;
    running += value;
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (base + pos < n) {
      out[base + pos] = tile[pos];
    }
  }
  if (threadIdx.x == 0) {
    blockSums[blockIdx.x] = blockTotal;
  }
}

__global__ void add_block_offsets_kernel(unsigned* out, int n, const unsigned* blockOffsets) {
  const unsigned offset = blockOffsets[blockIdx.x];
  const int base = blockIdx.x * TILE;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int index = base + coalesced_position(item);
    if (index < n) {
      out[index] += offset;
    }
  }
}

constexpr size_t SCAN_WORKSPACE_SLACK = 2;

static size_t scanWorkspace(int n) {
  size_t words = SCAN_WORKSPACE_SLACK;
  for (int blocks = cdiv(n, TILE); blocks > 1; blocks = cdiv(blocks, TILE)) {
    const size_t sumsAndScannedSums = 2 * blocks;
    words += sumsAndScannedSums;
  }
  return words + SCAN_WORKSPACE_SLACK;
}

static void exclusiveScan(const unsigned* in, unsigned* out, int n, unsigned* workspace) {
  if (n <= 0) {
    return;
  }

  const int blocks = cdiv(n, TILE);
  unsigned* sums = workspace;
  unsigned* sumsScanned = sums + blocks;
  unsigned* deeperWorkspace = sumsScanned + blocks;
  launch(scan_block_kernel, dim3(blocks), dim3(BLOCK_THREADS), in, out, n, sums);
  if (blocks > 1) {
    exclusiveScan(sums, sumsScanned, blocks, deeperWorkspace);
    launch(add_block_offsets_kernel, dim3(blocks), dim3(BLOCK_THREADS), out, n, sumsScanned);
  }
}

__host__ __device__ inline unsigned digit_of(unsigned key, int shift) {
  return shift < UNSIGNED_BITS ? (key >> shift) & (BUCKETS - 1u) : 0u;
}

__host__ __device__ inline unsigned bit_of(unsigned key, int bit) {
  return bit < UNSIGNED_BITS ? (key >> bit) & 1u : 0u;
}

__host__ __device__ inline int bucket_major_index(int bucket, int block, int numBlocks) {
  return bucket * numBlocks + block;
}

__host__ __device__ inline int block_major_index(int block, int bucket) {
  return block * BUCKETS + bucket;
}

__global__ void radix_local_sort_kernel(const unsigned* keysIn,
                                        const unsigned* valsIn,
                                        int n,
                                        int shift,
                                        unsigned* keysTmp,
                                        unsigned* valsTmp,
                                        unsigned* counts,
                                        unsigned* localStart) {
  __shared__ unsigned keys_S[2][TILE];
  __shared__ unsigned vals_S[2][TILE];
  __shared__ unsigned buf[2 * BLOCK_THREADS];
  __shared__ unsigned bucketStart[BUCKETS];
  __shared__ unsigned bucketEnd[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    const bool isRealElement = pos < tileCount;
    keys_S[0][pos] = isRealElement ? keysIn[tileBase + pos] : PADDING_KEY;
    vals_S[0][pos] = isRealElement ? valsIn[tileBase + pos] : 0u;
  }
  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    bucketStart[d] = 0;
    bucketEnd[d] = 0;
  }
  __syncthreads();

  int current = 0;
  for (int bitInDigit = 0; bitInDigit < RADIX_BITS; ++bitInDigit) {
    const int bit = shift + bitInDigit;
    const int next = 1 - current;
    unsigned myBits[ITEMS_PER_THREAD];
    unsigned myOnes = 0;
    for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
      myBits[item] = bit_of(keys_S[current][thread_owned_position(item)], bit);
      myOnes += myBits[item];
    }

    unsigned totalOnes;
    unsigned onesBefore = block_inclusive_scan(myOnes, buf, &totalOnes) - myOnes;
    const unsigned zeros = TILE - totalOnes;
    for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
      const unsigned pos = thread_owned_position(item);
      const unsigned dst = myBits[item] ? zeros + onesBefore : pos - onesBefore;
      onesBefore += myBits[item];
      keys_S[next][dst] = keys_S[current][pos];
      vals_S[next][dst] = vals_S[current][pos];
    }
    __syncthreads();
    current = next;
  }

  const unsigned* sortedKeys = keys_S[current];
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = thread_owned_position(item);
    if (pos < tileCount) {
      const unsigned d = digit_of(sortedKeys[pos], shift);
      const bool startsBucket = pos == 0 || digit_of(sortedKeys[pos - 1], shift) != d;
      const bool endsBucket = pos == tileCount - 1 || digit_of(sortedKeys[pos + 1], shift) != d;
      if (startsBucket) {
        bucketStart[d] = pos;
      }
      if (endsBucket) {
        bucketEnd[d] = pos + 1;
      }
    }
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (pos < tileCount) {
      keysTmp[tileBase + pos] = keys_S[current][pos];
      valsTmp[tileBase + pos] = vals_S[current][pos];
    }
  }
  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    counts[bucket_major_index(d, blockIdx.x, gridDim.x)] = bucketEnd[d] - bucketStart[d];
    localStart[block_major_index(blockIdx.x, d)] = bucketStart[d];
  }
}

__global__ void radix_scatter_coalesced_kernel(const unsigned* keysTmp,
                                               const unsigned* valsTmp,
                                               int n,
                                               int shift,
                                               const unsigned* offsets,
                                               const unsigned* localStart,
                                               unsigned* keysOut,
                                               unsigned* valsOut) {
  __shared__ int localToGlobal[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    const int globalStart = offsets[bucket_major_index(d, blockIdx.x, gridDim.x)];
    const int bucketLocalStart = localStart[block_major_index(blockIdx.x, d)];
    localToGlobal[d] = globalStart - bucketLocalStart;
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (pos < tileCount) {
      const unsigned key = keysTmp[tileBase + pos];
      const int dst = localToGlobal[digit_of(key, shift)] + pos;
      keysOut[dst] = key;
      valsOut[dst] = valsTmp[tileBase + pos];
    }
  }
}

struct RadixSortScratch {
  unsigned* otherKeys;
  unsigned* otherVals;
  unsigned* tileSortedKeys;
  unsigned* tileSortedVals;
  unsigned* counts;
  unsigned* offsets;
  unsigned* localStart;
  unsigned* workspace;
};

static unsigned* allocateDeviceWords(size_t count) {
  unsigned* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(unsigned)));
  return pointer;
}

static RadixSortScratch allocateScratch(int n) {
  const int tableEntries = BUCKETS * cdiv(n, TILE);
  RadixSortScratch scratch;
  scratch.otherKeys = allocateDeviceWords(n);
  scratch.otherVals = allocateDeviceWords(n);
  scratch.tileSortedKeys = allocateDeviceWords(n);
  scratch.tileSortedVals = allocateDeviceWords(n);
  scratch.counts = allocateDeviceWords(tableEntries);
  scratch.offsets = allocateDeviceWords(tableEntries);
  scratch.localStart = allocateDeviceWords(tableEntries);
  scratch.workspace = allocateDeviceWords(scanWorkspace(tableEntries));
  return scratch;
}

static void freeScratch(RadixSortScratch& scratch) {
  CHECK(cudaFree(scratch.otherKeys));
  CHECK(cudaFree(scratch.otherVals));
  CHECK(cudaFree(scratch.tileSortedKeys));
  CHECK(cudaFree(scratch.tileSortedVals));
  CHECK(cudaFree(scratch.counts));
  CHECK(cudaFree(scratch.offsets));
  CHECK(cudaFree(scratch.localStart));
  CHECK(cudaFree(scratch.workspace));
}

static void radixSortPass(const unsigned* keysIn,
                          const unsigned* valsIn,
                          unsigned* keysOut,
                          unsigned* valsOut,
                          int n,
                          int shift,
                          RadixSortScratch& scratch) {
  const unsigned blocks = cdiv(n, TILE);
  launch(radix_local_sort_kernel,
         dim3(blocks),
         dim3(BLOCK_THREADS),
         keysIn,
         valsIn,
         n,
         shift,
         scratch.tileSortedKeys,
         scratch.tileSortedVals,
         scratch.counts,
         scratch.localStart);
  exclusiveScan(scratch.counts, scratch.offsets, BUCKETS * blocks, scratch.workspace);
  launch(radix_scatter_coalesced_kernel,
         dim3(blocks),
         dim3(BLOCK_THREADS),
         scratch.tileSortedKeys,
         scratch.tileSortedVals,
         n,
         shift,
         scratch.offsets,
         scratch.localStart,
         keysOut,
         valsOut);
}

static int radixPasses(int keyBits) {
  return (keyBits + RADIX_BITS - 1) / RADIX_BITS;
}

static void radixSort(unsigned* keys, unsigned* vals, int n, int keyBits) {
  RadixSortScratch scratch = allocateScratch(n);
  unsigned* keysIn = keys;
  unsigned* valsIn = vals;
  unsigned* keysOut = scratch.otherKeys;
  unsigned* valsOut = scratch.otherVals;
  for (int shift = 0; shift < keyBits; shift += RADIX_BITS) {
    radixSortPass(keysIn, valsIn, keysOut, valsOut, n, shift, scratch);
    std::swap(keysIn, keysOut);
    std::swap(valsIn, valsOut);
  }

  const bool resultIsInScratch = keysIn != keys;
  if (resultIsInScratch) {
    CHECK(cudaMemcpy(keys, keysIn, n * sizeof(unsigned), cudaMemcpyDeviceToDevice));
    CHECK(cudaMemcpy(vals, valsIn, n * sizeof(unsigned), cudaMemcpyDeviceToDevice));
  }
  CHECK(cudaGetLastError());
  CHECK(cudaDeviceSynchronize());
  freeScratch(scratch);
}

#ifndef CPU_TRIPS
#define CPU_TRIPS (1 << 15)
#endif

constexpr int DEFAULT_TRIPS = BY_BACKEND(1 << 22, CPU_TRIPS);
constexpr uint64_t TRIPS_SEED = 42;
constexpr int STATION_COUNT = 48;
constexpr int BIKE_COUNT = 1500;
constexpr int MS_PER_SECOND = 1000;
constexpr int MS_PER_MINUTE = 60 * MS_PER_SECOND;
constexpr int MS_PER_HOUR = 60 * MS_PER_MINUTE;
constexpr int SERVICE_OPENS_MS = 6 * MS_PER_HOUR;
constexpr int SERVICE_CLOSES_MS = 23 * MS_PER_HOUR;
constexpr int MORNING_RUSH_MS = 8 * MS_PER_HOUR;
constexpr int EVENING_RUSH_MS = 18 * MS_PER_HOUR;
constexpr int RUSH_HALF_WIDTH_MS = MS_PER_HOUR;
constexpr int COMMUTE_MS_MIN = 4 * MS_PER_MINUTE;
constexpr int COMMUTE_MS_MAX = 25 * MS_PER_MINUTE;
constexpr int LEISURE_MS_MIN = 10 * MS_PER_MINUTE;
constexpr int LEISURE_MS_MAX = 70 * MS_PER_MINUTE;
constexpr int LEISURE_PERCENT_OFF_PEAK = 50;
constexpr int ONE_FULL_STATION_IN = 25;
constexpr int DOCK_SEARCH_MS_MIN = 5 * MS_PER_MINUTE;
constexpr int DOCK_SEARCH_MS_MAX = 40 * MS_PER_MINUTE;
constexpr int INCLUDED_MS = 45 * MS_PER_MINUTE;
constexpr int BASE_FARE_CENTS = 300;
constexpr int CENTS_PER_STARTED_BLOCK = 100;
constexpr int FARE_BLOCK_MS = 10 * MS_PER_MINUTE;
constexpr double CENTS_PER_BOLIVIANO = 100.0;
constexpr int LONGEST_TRIPS_SHOWN = 5;
constexpr double HUNDRED_PERCENT = 100.0;

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int between(int lowest, int highest) {
    const uint64_t span = static_cast<uint64_t>(highest - lowest) + 1;
    return lowest + static_cast<int>(next() % span);
  }

  bool oneIn(int n) {
    return next() % static_cast<uint64_t>(n) == 0;
  }

  bool percentChance(int percent) {
    return between(0, 99) < percent;
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

enum class TripKind { Commute, Leisure };

struct Trip {
  int tripId;
  int bikeId;
  int startStation;
  int endStation;
  int startMs;
  int endMs;
  int fareCents;
};

struct TripDurations {
  std::vector<unsigned> durationMs;
  std::vector<unsigned> tripId;
};

static int durationOf(const Trip& trip) {
  return trip.endMs - trip.startMs;
}

static int fareCents(int durationMs) {
  const int startedBlocks = (durationMs + FARE_BLOCK_MS - 1) / FARE_BLOCK_MS;
  return BASE_FARE_CENTS + CENTS_PER_STARTED_BLOCK * startedBlocks;
}

static bool isLate(unsigned durationMs) {
  return durationMs > static_cast<unsigned>(INCLUDED_MS);
}

static bool isRushHour(int timeMs) {
  const bool morning = std::abs(timeMs - MORNING_RUSH_MS) < RUSH_HALF_WIDTH_MS;
  const bool evening = std::abs(timeMs - EVENING_RUSH_MS) < RUSH_HALF_WIDTH_MS;
  return morning || evening;
}

static TripKind pickKind(SplitMix64& rng, int startMs) {
  if (isRushHour(startMs)) {
    return TripKind::Commute;
  }
  return rng.percentChance(LEISURE_PERCENT_OFF_PEAK) ? TripKind::Leisure : TripKind::Commute;
}

static int roundDownToSecond(int ms) {
  return ms / MS_PER_SECOND * MS_PER_SECOND;
}

static int rideMs(SplitMix64& rng, TripKind kind) {
  int ms = kind == TripKind::Commute ? rng.between(COMMUTE_MS_MIN, COMMUTE_MS_MAX)
                                     : rng.between(LEISURE_MS_MIN, LEISURE_MS_MAX);
  const bool endStationWasFull = rng.oneIn(ONE_FULL_STATION_IN);
  if (endStationWasFull) {
    ms += rng.between(DOCK_SEARCH_MS_MIN, DOCK_SEARCH_MS_MAX);
  }
  return roundDownToSecond(ms);
}

static Trip makeTrip(SplitMix64& rng, int tripId) {
  Trip trip;
  trip.tripId = tripId;
  trip.bikeId = rng.between(0, BIKE_COUNT - 1);
  trip.startStation = rng.between(0, STATION_COUNT - 1);
  trip.endStation = rng.between(0, STATION_COUNT - 1);
  trip.startMs = roundDownToSecond(rng.between(SERVICE_OPENS_MS, SERVICE_CLOSES_MS));
  trip.endMs = trip.startMs + rideMs(rng, pickKind(rng, trip.startMs));
  trip.fareCents = fareCents(durationOf(trip));
  return trip;
}

static std::vector<Trip> makeTrips(int n) {
  SplitMix64 rng(TRIPS_SEED);
  std::vector<Trip> trips(n);
  for (int i = 0; i < n; ++i) {
    trips[i] = makeTrip(rng, i);
  }
  return trips;
}

static TripDurations durationsOf(const std::vector<Trip>& trips) {
  TripDurations durations;
  durations.durationMs.resize(trips.size());
  durations.tripId.resize(trips.size());
  for (size_t i = 0; i < trips.size(); ++i) {
    durations.durationMs[i] = durationOf(trips[i]);
    durations.tripId[i] = trips[i].tripId;
  }
  return durations;
}

static int bitsNeeded(unsigned maxKey) {
  int bits = 0;
  while (bits < UNSIGNED_BITS && (maxKey >> bits) != 0) {
    ++bits;
  }
  return imax(bits, 1);
}

struct DeviceSortResult {
  TripDurations sorted;
  double ms;
};

static DeviceSortResult sortOnDevice(const TripDurations& durations, int keyBits) {
  const int n = static_cast<int>(durations.durationMs.size());
  const size_t bytes = n * sizeof(unsigned);
  unsigned* deviceKeys = allocateDeviceWords(n);
  unsigned* deviceVals = allocateDeviceWords(n);
  CHECK(cudaMemcpy(deviceKeys, durations.durationMs.data(), bytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(deviceVals, durations.tripId.data(), bytes, cudaMemcpyHostToDevice));

  Timer timer;
  radixSort(deviceKeys, deviceVals, n, keyBits);
  const double ms = timer.ms();

  DeviceSortResult result{{std::vector<unsigned>(n), std::vector<unsigned>(n)}, ms};
  CHECK(cudaMemcpy(result.sorted.durationMs.data(), deviceKeys, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaMemcpy(result.sorted.tripId.data(), deviceVals, bytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(deviceKeys));
  CHECK(cudaFree(deviceVals));
  return result;
}

static unsigned percentile(const std::vector<unsigned>& sorted, double p) {
  const size_t rank = static_cast<size_t>(std::ceil(p / HUNDRED_PERCENT * sorted.size()));
  return sorted[rank == 0 ? 0 : rank - 1];
}

static size_t lateTripsInSorted(const std::vector<unsigned>& sorted) {
  const auto firstLate =
      std::upper_bound(sorted.begin(), sorted.end(), static_cast<unsigned>(INCLUDED_MS));
  return static_cast<size_t>(sorted.end() - firstLate);
}

static size_t lateTripsCountedOneByOne(const std::vector<Trip>& trips) {
  size_t late = 0;
  for (const Trip& trip : trips) {
    if (isLate(durationOf(trip))) {
      ++late;
    }
  }
  return late;
}

struct Percentile {
  double p;
  const char* name;
};

constexpr Percentile REPORTED_PERCENTILES[] = {
    {50, "median"}, {90, "p90"}, {95, "p95"}, {99, "p99"}};

static bool everyIdKeptItsKey(const TripDurations& input, const TripDurations& sorted) {
  for (size_t i = 0; i < sorted.durationMs.size(); ++i) {
    if (input.durationMs[sorted.tripId[i]] != sorted.durationMs[i]) {
      return false;
    }
  }
  return true;
}

static bool tiesKeepAscendingTripIds(const TripDurations& sorted) {
  for (size_t i = 1; i < sorted.durationMs.size(); ++i) {
    const bool tie = sorted.durationMs[i] == sorted.durationMs[i - 1];
    if (tie && sorted.tripId[i - 1] >= sorted.tripId[i]) {
      return false;
    }
  }
  return true;
}

static bool percentilesMatch(const std::vector<unsigned>& sorted,
                             const std::vector<unsigned>& reference) {
  for (const Percentile& row : REPORTED_PERCENTILES) {
    if (percentile(sorted, row.p) != percentile(reference, row.p)) {
      return false;
    }
  }
  return true;
}

static double bolivianos(int cents) {
  return cents / CENTS_PER_BOLIVIANO;
}

static void printStatistic(const char* name, unsigned ms) {
  std::printf("%-10s %12u %10.1f %10.2f\n",
              name,
              ms,
              ms / static_cast<double>(MS_PER_MINUTE),
              bolivianos(fareCents(ms)));
}

static void printStatistics(const TripDurations& sorted) {
  const std::vector<unsigned>& durationMs = sorted.durationMs;
  const int n = static_cast<int>(durationMs.size());
  double sum = 0;
  for (unsigned ms : durationMs) {
    sum += ms;
  }
  const double average = sum / n;

  std::printf("%-10s %12s %10s %10s\n", "statistic", "ms", "minutes", "fare (Bs)");
  for (const Percentile& row : REPORTED_PERCENTILES) {
    printStatistic(row.name, percentile(durationMs, row.p));
  }
  std::printf("%-10s %12.0f %10.1f\n", "average", average, average / MS_PER_MINUTE);
  printStatistic("max", durationMs[n - 1]);
}

static void printLateTrips(size_t late, size_t n) {
  std::printf("\ntrips over the %d included minutes: %zu of %zu (%.1f %%)\n",
              INCLUDED_MS / MS_PER_MINUTE,
              late,
              n,
              HUNDRED_PERCENT * late / n);
}

static void printLongestTrips(const TripDurations& sorted, const std::vector<Trip>& trips) {
  const int n = static_cast<int>(sorted.durationMs.size());
  const int lastShown = imax(0, n - LONGEST_TRIPS_SHOWN);
  std::printf("longest trips:\n");
  for (int i = n - 1; i >= lastShown; --i) {
    const Trip& trip = trips[sorted.tripId[i]];
    std::printf("  trip t%d on bike b%d, s%d -> s%d: %u ms (%.1f min, Bs %.2f)\n",
                trip.tripId,
                trip.bikeId,
                trip.startStation,
                trip.endStation,
                sorted.durationMs[i],
                sorted.durationMs[i] / static_cast<double>(MS_PER_MINUTE),
                bolivianos(trip.fareCents));
  }
}

static void printTimings(double cpuMs, double deviceMs) {
  std::printf("\n%-34s %10s\n", "step", "time (ms)");
  std::printf("%-34s %10.2f\n", "std::sort of the keys (CPU)", cpuMs);
  std::printf("%-34s %10.2f\n", "radix sort of the pairs (device)", deviceMs);
  std::printf("\n");
}

static bool reportCheck(const char* label, bool ok) {
  std::printf("%-37s%s\n", label, ok ? "PASS" : "FAIL");
  return ok;
}

struct CheckInputs {
  const std::vector<Trip>& trips;
  const TripDurations& input;
  const TripDurations& sorted;
  const std::vector<unsigned>& reference;
};

static bool runChecks(const CheckInputs& in) {
  const bool lateCountsMatch =
      lateTripsInSorted(in.sorted.durationMs) == lateTripsCountedOneByOne(in.trips);

  bool ok = true;
  ok = reportCheck("sorted keys equal std::sort:", in.sorted.durationMs == in.reference) && ok;
  ok = reportCheck("every trip id kept its key:", everyIdKeptItsKey(in.input, in.sorted)) && ok;
  ok =
      reportCheck("equal durations keep ascending ids:", tiesKeepAscendingTripIds(in.sorted)) && ok;
  ok = reportCheck("percentiles equal reference:",
                   percentilesMatch(in.sorted.durationMs, in.reference)) &&
       ok;
  ok = reportCheck("late trips equal a direct count:", lateCountsMatch) && ok;
  return ok;
}

int main(int argc, char** argv) {
  const int n = argc > 1 ? std::atoi(argv[1]) : DEFAULT_TRIPS;
  std::printf("Bike-share trip-duration percentiles with a radix sort on %s\n", CUDA_BACKEND);

  const std::vector<Trip> trips = makeTrips(n);
  const TripDurations durations = durationsOf(trips);
  const unsigned longestMs =
      *std::max_element(durations.durationMs.begin(), durations.durationMs.end());
  const int keyBits = bitsNeeded(longestMs);
  std::printf("%d trips, longest %u ms -> %d key bits -> %d radix passes of %d bits\n\n",
              n,
              longestMs,
              keyBits,
              radixPasses(keyBits),
              RADIX_BITS);

  std::vector<unsigned> reference = durations.durationMs;
  Timer cpuTimer;
  std::sort(reference.begin(), reference.end());
  const double cpuMs = cpuTimer.ms();

  const DeviceSortResult device = sortOnDevice(durations, keyBits);
  const TripDurations& sorted = device.sorted;

  printStatistics(sorted);
  printLateTrips(lateTripsInSorted(sorted.durationMs), sorted.durationMs.size());
  printLongestTrips(sorted, trips);
  printTimings(cpuMs, device.ms);

  const bool ok = runChecks({trips, durations, sorted, reference});
  std::printf("%s\n", ok ? "ALL PASS" : "SOME TESTS FAILED");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_trip_percentiles.cu -o bikeshare_trip_percentiles && ./bikeshare_trip_percentiles

## `sorting.cu`

In [ ]:
%%writefile sorting.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <cstdlib>
#include <random>
#include <utility>
#include <vector>

constexpr int BLOCK_THREADS = 128;
constexpr int ITEMS_PER_THREAD = 8;
constexpr int TILE = BLOCK_THREADS * ITEMS_PER_THREAD;
constexpr int UNSIGNED_BITS = 32;
constexpr unsigned PADDING_KEY = 0xFFFFFFFFu;

__device__ inline int coalesced_position(int item) {
  return item * BLOCK_THREADS + threadIdx.x;
}

__device__ inline int thread_owned_position(int item) {
  return threadIdx.x * ITEMS_PER_THREAD + item;
}

__device__ inline unsigned block_inclusive_scan(unsigned value, unsigned* buf, unsigned* total) {
  const int t = threadIdx.x;
  buf[t] = value;
  __syncthreads();

  int readHalf = 0;
  for (int stride = 1; stride < BLOCK_THREADS; stride *= 2) {
    const int writeHalf = 1 - readHalf;
    unsigned sum = buf[readHalf * BLOCK_THREADS + t];
    if (t >= stride) {
      sum += buf[readHalf * BLOCK_THREADS + t - stride];
    }
    buf[writeHalf * BLOCK_THREADS + t] = sum;
    readHalf = writeHalf;
    __syncthreads();
  }

  *total = buf[readHalf * BLOCK_THREADS + BLOCK_THREADS - 1];
  const unsigned inclusivePrefix = buf[readHalf * BLOCK_THREADS + t];
  __syncthreads();
  return inclusivePrefix;
}

__global__ void scan_block_kernel(const unsigned* in, unsigned* out, int n, unsigned* blockSums) {
  __shared__ unsigned tile[TILE];
  __shared__ unsigned buf[2 * BLOCK_THREADS];
  const int base = blockIdx.x * TILE;

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    tile[pos] = base + pos < n ? in[base + pos] : 0u;
  }
  __syncthreads();

  unsigned threadSum = 0;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    threadSum += tile[thread_owned_position(item)];
  }

  unsigned blockTotal;
  unsigned running = block_inclusive_scan(threadSum, buf, &blockTotal) - threadSum;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = thread_owned_position(item);
    const unsigned value = tile[pos];
    tile[pos] = running;
    running += value;
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (base + pos < n) {
      out[base + pos] = tile[pos];
    }
  }
  if (threadIdx.x == 0) {
    blockSums[blockIdx.x] = blockTotal;
  }
}

__global__ void add_block_offsets_kernel(unsigned* out, int n, const unsigned* blockOffsets) {
  const unsigned offset = blockOffsets[blockIdx.x];
  const int base = blockIdx.x * TILE;
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int index = base + coalesced_position(item);
    if (index < n) {
      out[index] += offset;
    }
  }
}

constexpr size_t SCAN_WORKSPACE_SLACK = 2;

static size_t scanWorkspace(int n) {
  size_t words = SCAN_WORKSPACE_SLACK;
  for (int blocks = cdiv(n, TILE); blocks > 1; blocks = cdiv(blocks, TILE)) {
    const size_t sumsAndScannedSums = 2 * blocks;
    words += sumsAndScannedSums;
  }
  return words + SCAN_WORKSPACE_SLACK;
}

static void exclusiveScan(const unsigned* in, unsigned* out, int n, unsigned* workspace) {
  if (n <= 0) {
    return;
  }

  const int blocks = cdiv(n, TILE);
  unsigned* sums = workspace;
  unsigned* sumsScanned = sums + blocks;
  unsigned* deeperWorkspace = sumsScanned + blocks;
  launch(scan_block_kernel, dim3(blocks), dim3(BLOCK_THREADS), in, out, n, sums);
  if (blocks > 1) {
    exclusiveScan(sums, sumsScanned, blocks, deeperWorkspace);
    launch(add_block_offsets_kernel, dim3(blocks), dim3(BLOCK_THREADS), out, n, sumsScanned);
  }
}

__host__ __device__ inline unsigned digit_of(unsigned key, int shift, int r) {
  return shift < UNSIGNED_BITS ? (key >> shift) & ((1u << r) - 1u) : 0u;
}

__host__ __device__ inline unsigned bit_of(unsigned key, int bit) {
  return digit_of(key, bit, 1);
}

__global__ void radix_bit_kernel(const unsigned* keys, unsigned* bits, int n, int bit) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    bits[i] = bit_of(keys[i], bit);
  }
}

__global__ void radix_scatter_kernel(const unsigned* keysIn,
                                     const unsigned* valsIn,
                                     const unsigned* bits,
                                     const unsigned* onesBefore,
                                     int n,
                                     unsigned* keysOut,
                                     unsigned* valsOut) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    const unsigned totalOnes = onesBefore[n - 1] + bits[n - 1];
    const unsigned zeros = n - totalOnes;
    const unsigned dst = bits[i] ? zeros + onesBefore[i] : i - onesBefore[i];
    keysOut[dst] = keysIn[i];
    valsOut[dst] = valsIn[i];
  }
}

__host__ __device__ inline int bucket_major_index(int bucket, int block, int numBlocks) {
  return bucket * numBlocks + block;
}

__host__ __device__ inline int block_major_index(int block, int bucket, int buckets) {
  return block * buckets + bucket;
}

template <int RADIX_BITS>
__global__ void radix_local_sort_kernel(const unsigned* keysIn,
                                        const unsigned* valsIn,
                                        int n,
                                        int shift,
                                        unsigned* keysTmp,
                                        unsigned* valsTmp,
                                        unsigned* counts,
                                        unsigned* localStart) {
  constexpr int BUCKETS = 1 << RADIX_BITS;
  __shared__ unsigned keys_S[2][TILE];
  __shared__ unsigned vals_S[2][TILE];
  __shared__ unsigned buf[2 * BLOCK_THREADS];
  __shared__ unsigned bucketStart[BUCKETS];
  __shared__ unsigned bucketEnd[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    const bool isRealElement = pos < tileCount;
    keys_S[0][pos] = isRealElement ? keysIn[tileBase + pos] : PADDING_KEY;
    vals_S[0][pos] = isRealElement ? valsIn[tileBase + pos] : 0u;
  }
  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    bucketStart[d] = 0;
    bucketEnd[d] = 0;
  }
  __syncthreads();

  int current = 0;
  for (int bitInDigit = 0; bitInDigit < RADIX_BITS; ++bitInDigit) {
    const int bit = shift + bitInDigit;
    const int next = 1 - current;
    unsigned myBits[ITEMS_PER_THREAD];
    unsigned myOnes = 0;
    for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
      myBits[item] = bit_of(keys_S[current][thread_owned_position(item)], bit);
      myOnes += myBits[item];
    }

    unsigned totalOnes;
    unsigned onesBefore = block_inclusive_scan(myOnes, buf, &totalOnes) - myOnes;
    const unsigned zeros = TILE - totalOnes;
    for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
      const unsigned pos = thread_owned_position(item);
      const unsigned dst = myBits[item] ? zeros + onesBefore : pos - onesBefore;
      onesBefore += myBits[item];
      keys_S[next][dst] = keys_S[current][pos];
      vals_S[next][dst] = vals_S[current][pos];
    }
    __syncthreads();
    current = next;
  }

  const unsigned* sortedKeys = keys_S[current];
  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = thread_owned_position(item);
    if (pos < tileCount) {
      const unsigned d = digit_of(sortedKeys[pos], shift, RADIX_BITS);
      const bool startsBucket = pos == 0 || digit_of(sortedKeys[pos - 1], shift, RADIX_BITS) != d;
      const bool endsBucket =
          pos == tileCount - 1 || digit_of(sortedKeys[pos + 1], shift, RADIX_BITS) != d;
      if (startsBucket) {
        bucketStart[d] = pos;
      }
      if (endsBucket) {
        bucketEnd[d] = pos + 1;
      }
    }
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (pos < tileCount) {
      keysTmp[tileBase + pos] = keys_S[current][pos];
      valsTmp[tileBase + pos] = vals_S[current][pos];
    }
  }
  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    counts[bucket_major_index(d, blockIdx.x, gridDim.x)] = bucketEnd[d] - bucketStart[d];
    localStart[block_major_index(blockIdx.x, d, BUCKETS)] = bucketStart[d];
  }
}

template <int RADIX_BITS>
__global__ void radix_scatter_coalesced_kernel(const unsigned* keysTmp,
                                               const unsigned* valsTmp,
                                               int n,
                                               int shift,
                                               const unsigned* offsets,
                                               const unsigned* localStart,
                                               unsigned* keysOut,
                                               unsigned* valsOut) {
  constexpr int BUCKETS = 1 << RADIX_BITS;
  __shared__ int localToGlobal[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int d = t; d < BUCKETS; d += BLOCK_THREADS) {
    const int globalStart = offsets[bucket_major_index(d, blockIdx.x, gridDim.x)];
    const int bucketLocalStart = localStart[block_major_index(blockIdx.x, d, BUCKETS)];
    localToGlobal[d] = globalStart - bucketLocalStart;
  }
  __syncthreads();

  for (int item = 0; item < ITEMS_PER_THREAD; ++item) {
    const int pos = coalesced_position(item);
    if (pos < tileCount) {
      const unsigned key = keysTmp[tileBase + pos];
      const int dst = localToGlobal[digit_of(key, shift, RADIX_BITS)] + pos;
      keysOut[dst] = key;
      valsOut[dst] = valsTmp[tileBase + pos];
    }
  }
}

__host__ __device__ inline int co_rank(int k, const unsigned* A, int m, const unsigned* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large = i > 0 && j < n && A[i - 1] > B[j];
    const bool i_too_small = j > 0 && i < m && B[j - 1] >= A[i];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__host__ __device__ inline void merge_sequential_kv(const unsigned* Ak,
                                                    const unsigned* Av,
                                                    int m,
                                                    const unsigned* Bk,
                                                    const unsigned* Bv,
                                                    int n,
                                                    unsigned* Ck,
                                                    unsigned* Cv) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    if (Ak[i] <= Bk[j]) {
      Ck[k] = Ak[i];
      Cv[k++] = Av[i++];
    } else {
      Ck[k] = Bk[j];
      Cv[k++] = Bv[j++];
    }
  }
  while (i < m) {
    Ck[k] = Ak[i];
    Cv[k++] = Av[i++];
  }
  while (j < n) {
    Ck[k] = Bk[j];
    Cv[k++] = Bv[j++];
  }
}

__host__ __device__ inline void stable_insertion_sort(unsigned* keys,
                                                      unsigned* vals,
                                                      int begin,
                                                      int end) {
  for (int i = begin + 1; i < end; ++i) {
    const unsigned key = keys[i];
    const unsigned val = vals[i];
    int j = i - 1;
    while (j >= begin && keys[j] > key) {
      keys[j + 1] = keys[j];
      vals[j + 1] = vals[j];
      --j;
    }
    keys[j + 1] = key;
    vals[j + 1] = val;
  }
}

__global__ void sort_runs_kernel(unsigned* keys, unsigned* vals, int n, int run) {
  const int runStart = (blockIdx.x * blockDim.x + threadIdx.x) * run;
  if (runStart < n) {
    const int runEnd = imin(runStart + run, n);
    stable_insertion_sort(keys, vals, runStart, runEnd);
  }
}

__global__ void merge_pass_kernel(const unsigned* keysIn,
                                  const unsigned* valsIn,
                                  unsigned* keysOut,
                                  unsigned* valsOut,
                                  int total,
                                  int width,
                                  int outputsPerThread) {
  const int tid = blockIdx.x * blockDim.x + threadIdx.x;
  const int sliceBegin = imin(tid * outputsPerThread, total);
  const int sliceEnd = imin(sliceBegin + outputsPerThread, total);

  int k = sliceBegin;
  while (k < sliceEnd) {
    const int pairStart = (k / (2 * width)) * (2 * width);
    const int mid = imin(pairStart + width, total);
    const int pairEnd = imin(pairStart + 2 * width, total);
    const int pieceEnd = imin(sliceEnd, pairEnd);

    const int m = mid - pairStart;
    const int n = pairEnd - mid;
    const int k_curr = k - pairStart;
    const int k_next = pieceEnd - pairStart;
    const int i_curr = co_rank(k_curr, keysIn + pairStart, m, keysIn + mid, n);
    const int i_next = co_rank(k_next, keysIn + pairStart, m, keysIn + mid, n);
    const int j_curr = k_curr - i_curr;
    const int j_next = k_next - i_next;
    merge_sequential_kv(keysIn + pairStart + i_curr,
                        valsIn + pairStart + i_curr,
                        i_next - i_curr,
                        keysIn + mid + j_curr,
                        valsIn + mid + j_curr,
                        j_next - j_curr,
                        keysOut + k,
                        valsOut + k);

    k = pieceEnd;
  }
}

constexpr int SIMPLE_KERNEL_THREADS = 256;
constexpr int MAX_RADIX_BITS = 8;
constexpr int MAX_BUCKETS = 1 << MAX_RADIX_BITS;
constexpr int MERGE_SORT_RUN = 16;
constexpr int MERGE_OUTPUTS_PER_THREAD = 64;

struct SortBuffers {
  unsigned* keys[2];
  unsigned* vals[2];
  unsigned* bits;
  unsigned* scan;
  unsigned* counts;
  unsigned* offsets;
  unsigned* localStart;
  unsigned* workspace;
};

static unsigned* allocateDeviceWords(size_t count) {
  unsigned* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(unsigned)));
  return pointer;
}

static SortBuffers allocateBuffers(int n) {
  const int tableEntries = MAX_BUCKETS * cdiv(n, TILE);
  SortBuffers buffers;
  for (int side = 0; side < 2; ++side) {
    buffers.keys[side] = allocateDeviceWords(n);
    buffers.vals[side] = allocateDeviceWords(n);
  }
  buffers.bits = allocateDeviceWords(n);
  buffers.scan = allocateDeviceWords(n);
  buffers.counts = allocateDeviceWords(tableEntries);
  buffers.offsets = allocateDeviceWords(tableEntries);
  buffers.localStart = allocateDeviceWords(tableEntries);
  buffers.workspace = allocateDeviceWords(scanWorkspace(imax(n, tableEntries)));
  return buffers;
}

static void freeBuffers(SortBuffers& buffers) {
  for (int side = 0; side < 2; ++side) {
    CHECK(cudaFree(buffers.keys[side]));
    CHECK(cudaFree(buffers.vals[side]));
  }
  CHECK(cudaFree(buffers.bits));
  CHECK(cudaFree(buffers.scan));
  CHECK(cudaFree(buffers.counts));
  CHECK(cudaFree(buffers.offsets));
  CHECK(cudaFree(buffers.localStart));
  CHECK(cudaFree(buffers.workspace));
}

static int otherSide(int side) {
  return 1 - side;
}

static int radixSortNaive(SortBuffers& buffers, int n, int keyBits) {
  const unsigned blocks = cdiv(n, SIMPLE_KERNEL_THREADS);
  int source = 0;
  for (int bit = 0; bit < keyBits; ++bit) {
    const int destination = otherSide(source);
    launch(radix_bit_kernel,
           dim3(blocks),
           dim3(SIMPLE_KERNEL_THREADS),
           buffers.keys[source],
           buffers.bits,
           n,
           bit);
    exclusiveScan(buffers.bits, buffers.scan, n, buffers.workspace);
    launch(radix_scatter_kernel,
           dim3(blocks),
           dim3(SIMPLE_KERNEL_THREADS),
           buffers.keys[source],
           buffers.vals[source],
           buffers.bits,
           buffers.scan,
           n,
           buffers.keys[destination],
           buffers.vals[destination]);
    source = destination;
  }
  return source;
}

template <int RADIX_BITS>
static int radixSortLocal(SortBuffers& buffers, int n, int keyBits) {
  static_assert(RADIX_BITS <= MAX_RADIX_BITS, "the bucket table is sized for MAX_RADIX_BITS");
  constexpr int BUCKETS = 1 << RADIX_BITS;
  const unsigned blocks = cdiv(n, TILE);
  unsigned* tileSortedKeys = buffers.bits;
  unsigned* tileSortedVals = buffers.scan;
  int source = 0;
  for (int shift = 0; shift < keyBits; shift += RADIX_BITS) {
    const int destination = otherSide(source);
    launch(radix_local_sort_kernel<RADIX_BITS>,
           dim3(blocks),
           dim3(BLOCK_THREADS),
           buffers.keys[source],
           buffers.vals[source],
           n,
           shift,
           tileSortedKeys,
           tileSortedVals,
           buffers.counts,
           buffers.localStart);
    exclusiveScan(buffers.counts, buffers.offsets, BUCKETS * blocks, buffers.workspace);
    launch(radix_scatter_coalesced_kernel<RADIX_BITS>,
           dim3(blocks),
           dim3(BLOCK_THREADS),
           tileSortedKeys,
           tileSortedVals,
           n,
           shift,
           buffers.offsets,
           buffers.localStart,
           buffers.keys[destination],
           buffers.vals[destination]);
    source = destination;
  }
  return source;
}

static int mergeSort(SortBuffers& buffers, int n) {
  const unsigned runs = cdiv(n, MERGE_SORT_RUN);
  launch(sort_runs_kernel,
         dim3(cdiv(runs, SIMPLE_KERNEL_THREADS)),
         dim3(SIMPLE_KERNEL_THREADS),
         buffers.keys[0],
         buffers.vals[0],
         n,
         MERGE_SORT_RUN);

  const unsigned mergeThreads = cdiv(n, MERGE_OUTPUTS_PER_THREAD);
  int source = 0;
  for (int width = MERGE_SORT_RUN; width < n; width *= 2) {
    const int destination = otherSide(source);
    launch(merge_pass_kernel,
           dim3(cdiv(mergeThreads, SIMPLE_KERNEL_THREADS)),
           dim3(SIMPLE_KERNEL_THREADS),
           buffers.keys[source],
           buffers.vals[source],
           buffers.keys[destination],
           buffers.vals[destination],
           n,
           width,
           MERGE_OUTPUTS_PER_THREAD);
    source = destination;
  }
  return source;
}

enum class SortVariant { Radix1Naive, Radix1Local, Radix4Local, Radix8Local, MergeSort };

constexpr SortVariant ALL_VARIANTS[] = {SortVariant::Radix1Naive,
                                        SortVariant::Radix1Local,
                                        SortVariant::Radix4Local,
                                        SortVariant::Radix8Local,
                                        SortVariant::MergeSort};

static const char* variantName(SortVariant variant) {
  switch (variant) {
    case SortVariant::Radix1Naive:
      return "radix 1 bit, naive scatter";
    case SortVariant::Radix1Local:
      return "radix 1 bit, local sort";
    case SortVariant::Radix4Local:
      return "radix 4 bits, local sort";
    case SortVariant::Radix8Local:
      return "radix 8 bits, local sort";
    case SortVariant::MergeSort:
      return "merge sort (co-rank merge)";
  }
  return "unknown";
}

static int runVariant(SortVariant variant, SortBuffers& buffers, int n, int keyBits) {
  int resultSide = 0;
  switch (variant) {
    case SortVariant::Radix1Naive:
      resultSide = radixSortNaive(buffers, n, keyBits);
      break;
    case SortVariant::Radix1Local:
      resultSide = radixSortLocal<1>(buffers, n, keyBits);
      break;
    case SortVariant::Radix4Local:
      resultSide = radixSortLocal<4>(buffers, n, keyBits);
      break;
    case SortVariant::Radix8Local:
      resultSide = radixSortLocal<8>(buffers, n, keyBits);
      break;
    case SortVariant::MergeSort:
      resultSide = mergeSort(buffers, n);
      break;
  }
  CHECK(cudaGetLastError());
  CHECK(cudaDeviceSynchronize());
  return resultSide;
}

struct KeyValuePairs {
  std::vector<unsigned> keys;
  std::vector<unsigned> vals;
};

static bool operator==(const KeyValuePairs& a, const KeyValuePairs& b) {
  return a.keys == b.keys && a.vals == b.vals;
}

static void copyToDevice(unsigned* destination, const std::vector<unsigned>& source) {
  CHECK(cudaMemcpy(
      destination, source.data(), source.size() * sizeof(unsigned), cudaMemcpyHostToDevice));
}

static void copyToHost(std::vector<unsigned>& destination, const unsigned* source) {
  CHECK(cudaMemcpy(
      destination.data(), source, destination.size() * sizeof(unsigned), cudaMemcpyDeviceToHost));
}

static double sortOnDevice(SortVariant variant,
                           SortBuffers& buffers,
                           KeyValuePairs& pairs,
                           int keyBits) {
  const int n = static_cast<int>(pairs.keys.size());
  copyToDevice(buffers.keys[0], pairs.keys);
  copyToDevice(buffers.vals[0], pairs.vals);

  Timer timer;
  const int resultSide = runVariant(variant, buffers, n, keyBits);
  const double ms = timer.ms();

  copyToHost(pairs.keys, buffers.keys[resultSide]);
  copyToHost(pairs.vals, buffers.vals[resultSide]);
  return ms;
}

static KeyValuePairs randomKeysWithPositions(int n, int keyBits, unsigned seed) {
  std::mt19937 rng(seed);
  const unsigned keyMask = keyBits >= UNSIGNED_BITS ? ~0u : (1u << keyBits) - 1u;
  KeyValuePairs pairs;
  pairs.keys.resize(n);
  pairs.vals.resize(n);
  for (int i = 0; i < n; ++i) {
    pairs.keys[i] = rng() & keyMask;
    pairs.vals[i] = i;
  }
  return pairs;
}

static KeyValuePairs stableSortOnHost(const std::vector<unsigned>& keys) {
  const int n = static_cast<int>(keys.size());
  std::vector<std::pair<unsigned, unsigned>> keyPosition(n);
  for (int i = 0; i < n; ++i) {
    keyPosition[i] = {keys[i], static_cast<unsigned>(i)};
  }
  std::stable_sort(keyPosition.begin(), keyPosition.end(), [](const auto& a, const auto& b) {
    return a.first < b.first;
  });

  KeyValuePairs sorted;
  sorted.keys.resize(n);
  sorted.vals.resize(n);
  for (int i = 0; i < n; ++i) {
    sorted.keys[i] = keyPosition[i].first;
    sorted.vals[i] = keyPosition[i].second;
  }
  return sorted;
}

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

constexpr int SMALL_CASE_CAPACITY = 3000;
constexpr int SMALL_CASE_KEY_BITS = 6;
constexpr unsigned SMALL_CASE_SEED_BASE = 100;

static bool checkSmallCases() {
  const int sizes[] = {1, 2, 1000, 2500};
  SortBuffers buffers = allocateBuffers(SMALL_CASE_CAPACITY);
  int passed = 0;
  int total = 0;
  for (int size : sizes) {
    const KeyValuePairs input =
        randomKeysWithPositions(size, SMALL_CASE_KEY_BITS, SMALL_CASE_SEED_BASE + size);
    const KeyValuePairs expected = stableSortOnHost(input.keys);
    for (SortVariant variant : ALL_VARIANTS) {
      KeyValuePairs pairs = input;
      sortOnDevice(variant, buffers, pairs, SMALL_CASE_KEY_BITS);
      const bool ok = pairs == expected;
      passed += ok;
      ++total;
      if (!ok) {
        std::printf("  FAIL: %s with n = %d\n", variantName(variant), size);
      }
    }
  }
  freeBuffers(buffers);

  const bool allPassed = passed == total;
  std::printf("small cases: %d/%d %s\n\n", passed, total, passOrFail(allPassed));
  return allPassed;
}

constexpr int DEFAULT_PAIRS = BY_BACKEND(1 << 24, 1 << 14);
constexpr int TIMED_KEY_BITS = BY_BACKEND(32, 16);
constexpr unsigned TIMED_SEED = 7;

constexpr double MS_PER_SECOND = 1e3;
constexpr double PAIRS_PER_MILLION = 1e6;

static double millionPairsPerSecond(int n, double ms) {
  const double seconds = ms / MS_PER_SECOND;
  return n / seconds / PAIRS_PER_MILLION;
}

static void printHostRow(const char* label, int n, double ms, const char* check) {
  std::printf("%-30s %10.2f %12.2f  %s\n", label, ms, millionPairsPerSecond(n, ms), check);
}

static void printDeviceRow(const char* label, int n, double ms, bool keysOk, bool stable) {
  std::printf("%-30s %10.2f %12.2f  %s, %s\n",
              label,
              ms,
              millionPairsPerSecond(n, ms),
              passOrFail(keysOk),
              stable ? "stable" : "NOT STABLE");
}

static bool runTimedComparison(int n, int keyBits) {
  const KeyValuePairs input = randomKeysWithPositions(n, keyBits, TIMED_SEED);

  std::vector<unsigned> keysOnly = input.keys;
  Timer sortTimer;
  std::sort(keysOnly.begin(), keysOnly.end());
  const double sortMs = sortTimer.ms();

  Timer stableTimer;
  const KeyValuePairs expected = stableSortOnHost(input.keys);
  const double stableMs = stableTimer.ms();

  SortBuffers buffers = allocateBuffers(n);
  KeyValuePairs warmUp = input;
  sortOnDevice(SortVariant::MergeSort, buffers, warmUp, keyBits);

  std::printf("sorting n = %d (key, value) pairs, keys of %d bits\n", n, keyBits);
  std::printf("%-30s %10s %12s  %s\n", "variant", "time (ms)", "Mpairs/s", "check (keys, stable)");
  printHostRow("std::sort (CPU, keys only)", n, sortMs, "-");
  printHostRow("std::stable_sort (CPU, pairs)", n, stableMs, "reference");

  bool allPassed = true;
  for (SortVariant variant : ALL_VARIANTS) {
    KeyValuePairs pairs = input;
    const double ms = sortOnDevice(variant, buffers, pairs, keyBits);
    const bool keysOk = pairs.keys == expected.keys;
    const bool stable = pairs.vals == expected.vals;
    allPassed = allPassed && keysOk && stable;
    printDeviceRow(variantName(variant), n, ms, keysOk, stable);
  }
  freeBuffers(buffers);
  return allPassed;
}

int main(int argc, char** argv) {
  const int n = argc > 1 ? std::atoi(argv[1]) : DEFAULT_PAIRS;
  std::printf("Parallel sorting (chapter 14) on %s\n\n", CUDA_BACKEND);

  const bool smallCasesOk = checkSmallCases();
  const bool timedOk = runTimedComparison(n, TIMED_KEY_BITS);

  const bool allOk = smallCasesOk && timedOk;
  std::printf("\n%s\n", allOk ? "ALL PASS" : "SOME TESTS FAILED");
  return allOk ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. sorting.cu -o sorting && ./sorting